# 🔴 Solution: Convex Hull (Graham Scan)

**Algorithm:** Graham scan — O(N log N)

**Reduction:** Find pivot (bottommost-leftmost), sort remaining points by polar angle via `np.arctan2`, then scan maintaining a stack where we pop on every clockwise turn (`cross ≤ 0`).

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# algorithm: Graham scan O(N log N)

import numpy as np

def convex_hull(points):
    pts = np.array(points, dtype=float)
    n = len(pts)
    if n <= 2:
        return pts.copy()
    # Pivot: lowest y, tie-break leftmost x
    pivot_idx = np.lexsort((pts[:, 0], pts[:, 1]))[0]
    pivot = pts[pivot_idx]
    others = np.delete(pts, pivot_idx, axis=0)
    dx = others[:, 0] - pivot[0]; dy = others[:, 1] - pivot[1]
    angles = np.arctan2(dy, dx); dists = np.hypot(dx, dy)
    order = np.lexsort((dists, angles))
    sorted_pts = np.vstack([pivot, others[order]])

    def cross(o, a, b):
        return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])

    stack = list(sorted_pts[:2])
    for p in sorted_pts[2:]:
        while len(stack) > 1 and cross(stack[-2], stack[-1], p) <= 0:
            stack.pop()
        stack.append(p)
    return np.array(stack)

In [ ]:
# 🔍 Verify solution
# Square + interior point
pts = np.array([[0.,0.],[1.,0.],[1.,1.],[0.,1.],[0.5,0.5]])
hull = convex_hull(pts)
print("Square hull:", hull)          # expect 4 points, area=1

# Triangle
tri = np.array([[0.,0.],[3.,0.],[1.5,2.]])
print("Triangle hull:", convex_hull(tri))  # expect all 3

In [ ]:
# ✅ Inline test suite
import numpy as np, time

def _area(p):
    p = np.array(p, dtype=float)
    x,y=p[:,0],p[:,1]
    return 0.5*abs(np.sum(x*np.roll(y,-1)-np.roll(x,-1)*y))

# ── Test 1: unit square + interior ────────────────────────────────────────
pts = np.array([[0.,0.],[1.,0.],[1.,1.],[0.,1.],[0.5,0.5]])
h = np.array(convex_hull(pts), dtype=float)
assert len(h)==4, f"Square hull: {len(h)} points"
assert abs(_area(h)-1.0)<1e-9, f"Square area: {_area(h)}"
print("Test 1 passed: square (interior filtered)")

# ── Test 2: triangle — all 3 on hull ──────────────────────────────────────
tri = np.array([[0.,0.],[4.,0.],[2.,3.]])
h2 = np.array(convex_hull(tri), dtype=float)
assert len(h2)==3, f"Triangle hull: {len(h2)}"
print("Test 2 passed: triangle")

# ── Test 3: square with random interior ────────────────────────────────────
rng=np.random.default_rng(99)
corners = np.array([[0.,0.],[10.,0.],[10.,10.],[0.,10.]])
interior = rng.uniform(0.5,9.5,(20,2))
pts3 = np.vstack([corners, interior])
h3 = np.array(convex_hull(pts3), dtype=float)
assert len(h3)==4, f"Square hull should have 4 points, got {len(h3)}"
assert abs(_area(h3)-100.)<1e-6, f"Area: {_area(h3)}"
print("Test 3 passed: 20 interior points filtered")

# ── Test 4: hexagon ────────────────────────────────────────────────────────
a=np.linspace(0,2*np.pi,6,endpoint=False)
hex_pts = np.stack([np.cos(a),np.sin(a)],axis=1)
rng2=np.random.default_rng(0)
pts4 = np.vstack([hex_pts, rng2.uniform(-0.4,0.4,(10,2))])
h4 = np.array(convex_hull(pts4), dtype=float)
assert len(h4)==6, f"Hexagon hull should have 6 points, got {len(h4)}"
print("Test 4 passed: hexagon")

# ── Test 5: N=50000 ───────────────────────────────────────────────────────
rng3=np.random.default_rng(42); pts5=rng3.standard_normal((50000,2))
t0=time.time(); h5=np.array(convex_hull(pts5),dtype=float); elapsed=time.time()-t0
assert h5.ndim==2 and h5.shape[1]==2 and len(h5)>=3
assert elapsed < 5.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: N=50000 ({elapsed:.3f}s)")

print("\nAll tests passed!")